In [1]:
import pandas as pd
import numpy as np
import random
import time

# Read input data
data = pd.read_excel('data/input42u.xlsx')
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# Read power demand data
power_demand_data = pd.read_excel('data/power_demand42u.xlsx')
power_demand = power_demand_data['load'].values

# PSO Parameters
num_particles = 50
num_iterations = 100
w = 0.5  # inertia weight
c1 = 1.5  # cognitive weight
c2 = 1.5  # social weight

# Define PSO-based power calculation
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown):
    num_units = len(pmin)

    # Initialize particles
    particles = np.random.uniform(pmin, pmax, (num_particles, num_units))
    velocities = np.random.uniform(-1, 1, (num_particles, num_units))

    # Initialize personal and global bests
    personal_best_positions = particles.copy()
    personal_best_scores = np.full(num_particles, float('inf'))
    global_best_position = np.zeros(num_units)
    global_best_score = float('inf')

    for iteration in range(num_iterations):
        for i in range(num_particles):
            # Ensure particles respect ramp rate constraints
            if iteration > 0:
                diff = particles[i] - personal_best_positions[i]
                ramp_up_mask = diff > 0
                ramp_down_mask = diff < 0
                ramp_up_values = np.minimum(diff[ramp_up_mask], rampup[ramp_up_mask])
                ramp_down_values = np.maximum(diff[ramp_down_mask], -rampdown[ramp_down_mask])
                particles[i][ramp_up_mask] += ramp_up_values
                particles[i][ramp_down_mask] += ramp_down_values

            # Ensure particles respect generation limits
            particles[i] = np.clip(particles[i], pmin, pmax)

            # Calculate fuel cost for the particle
            squared_P = np.multiply(particles[i], particles[i])
            fuel_cost = np.sum(np.add(a, np.add(np.multiply(b, particles[i]), np.multiply(c, squared_P))))

            # Penalty for demand mismatch
            power_output = np.sum(particles[i])
            mismatch_penalty = abs(demand - power_output) * 10000  # Large penalty for mismatch
            score = fuel_cost + mismatch_penalty

            # Update personal best
            if score < personal_best_scores[i]:
                personal_best_scores[i] = score
                personal_best_positions[i] = particles[i].copy()

            # Update global best
            if score < global_best_score:
                global_best_score = score
                global_best_position = particles[i].copy()

        # Update velocities and positions
        for i in range(num_particles):
            r1 = np.random.uniform(size=num_units)
            r2 = np.random.uniform(size=num_units)
            velocities[i] = (
                w * velocities[i] +
                c1 * r1 * (personal_best_positions[i] - particles[i]) +
                c2 * r2 * (global_best_position - particles[i])
            )
            particles[i] += velocities[i]

        # Normalization to meet demand
        total_power = np.sum(global_best_position)
        if total_power != 0:
            scaling_factor = demand / total_power
            global_best_position = global_best_position * scaling_factor

        # Ensure global best respects generation limits
        global_best_position = np.clip(global_best_position, pmin, pmax)

        # Termination condition for power balance
        if abs(demand - np.sum(global_best_position)) < 0.0001:
            break

    # Return the best solution
    best_power = global_best_position
    return best_power, iteration + 1

# Fuel cost calculation
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.divide(np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP))), 1000)
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# Loop through power demand data and calculate results
all_outputs = []
schedulling = []

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values
    P, iterations = calculate_power(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    # Output Results
    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost: Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({'Demand': demand, 
                        'Output': output, 
                        'Total Power Produced': np.sum(P),
                        'Total Fuel Cost': total_Fuel_Cost,  # No need to sum again, it's already a single value
                        'Computation Time': end_time - start_time})

    # Scheduling
    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost'] = total_Fuel_Cost  # Assign total cost directly
    schedulling.append(output_info)

# Calculate ramp rates and save to an additional sheet
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    # Check for ramp rate violations
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = violations[unit_index]
               
        # Check for generation limit violations
        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = gen_violation
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)
# Save results to Excel
output_file_path = 'PSO (ED).xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(all_outputs).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)
print("All results saved to 'PSO (ED).xlsx'")

Load Counter: 1
Output for Power Demand 8761.42 MW
    Unit  Power Produced (MW)
0      1           271.400630
1      2           371.500000
2      3           271.400630
3      4           271.400630
4      5           442.925829
5      6           442.925829
6      7           442.925829
7      8            94.990221
8      9            94.990221
9     10            94.990221
10    11            94.990221
11    12            43.424101
12    13            80.000000
13    14            81.420189
14    15           170.000000
15    16           238.832555
16    17           238.832555
17    18           238.832555
18    19            16.000000
19    20            16.000000
20    21            65.136151
21    22           118.500000
22    23            65.136151
23    24            65.136151
24    25            65.136151
25    26           118.500000
26    27            65.136151
27    28           118.500000
28    29           434.241008
29    30           434.241008
30    31           